# 02 — Preprocessing & Feature Engineering

**Tujuan.** Mengubah 9 tabel relasional mentah menjadi satu matriks fitur level
`shipment_id` yang siap dilatih, lalu menyimpannya ke `data/processed/` agar notebook
model dapat memuatnya tanpa mengulang komputasi.

**Prinsip anti-leakage.**
- Agregasi tabel anak hanya memakai kolom fitur (bukan target).
- Encoding kategorikal memakai kamus gabungan train+test, tetapi **tanpa** target.
- Target encoding dihitung out-of-fold (dibahas di notebook 04).

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
%matplotlib inline

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

# root proyek = folder terdekat ke atas yang berisi src/ dan data/
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(BASE))

import gc, time

from src.data_loader import PROCESSED_DIR, TRAIN_DIR, TEST_DIR, TARGET, CLASSES, path_of
from src.features import (DROP_COLS, TE_COLS, scan_feats, route_feats, inv_feats,
                          build_features, encode_categoricals)
from src.utils import set_seed

set_seed()
print("output fitur ->", PROCESSED_DIR)

output fitur -> C:\Datavidia\TASK-1\data\processed


## 1. Agregasi tabel anak (polars)

| Tabel | Agregasi ke level `shipment_id` |
|---|---|
| `scan_events` | jumlah scan, jumlah hub/device unik, statistik `gap_hours` (mean/std/min/max/sum/q90), statistik `scan_quality`, pangsa tiap `event_type` dan `device_type`, rata-rata kualitas & jeda khusus `upaya_antar`/`selesai` |
| `route_segments` | jumlah segmen, jarak/durasi/biaya (sum/mean/max/std), pangsa tiap `moda`, hub pertama & terakhir, durasi per km, biaya per km |
| `invoices` | jumlah invoice, jumlah jenis, nominal & diskon (sum/mean/max), pajak, `days_to_pay`, jumlah invoice `klaim`/`langganan`/`pengiriman`, status bayar `lunas`/`tertunda`/`gagal` |

Tabel master (`customers`, `couriers`, `hubs`, `service_packages`, `promotions`) di-join
langsung dengan prefiks agar tidak ada nama kolom bertabrakan.

In [2]:
# ---- bangun seluruh fitur untuk train & test ----
# Set True untuk membangun ulang dari CSV mentah; False memakai cache parquet.
FORCE_REBUILD = False

f_train = PROCESSED_DIR / "feat_train.parquet"
f_test = PROCESSED_DIR / "feat_test.parquet"

if FORCE_REBUILD or not (f_train.exists() and f_test.exists()):
    t0 = time.time()
    build_features("train")
    gc.collect()
    build_features("test")
    gc.collect()
    print(f"\nfeature engineering selesai dalam {time.time() - t0:.1f} detik")
else:
    print("cache fitur ditemukan, memakai parquet yang sudah ada")

for name in ["feat_train.parquet", "feat_test.parquet"]:
    p = PROCESSED_DIR / name
    n_rows = pl.scan_parquet(p).select(pl.len()).collect().item()
    print(f"{name:20s} {n_rows:>9,} baris  {len(pl.read_parquet_schema(p)):3d} kolom  "
          f"{p.stat().st_size/1e6:.0f} MB")

train -> (1243617, 199)  disimpan ke processed\feat_train.parquet
test  -> (362245, 198)  disimpan ke processed\feat_test.parquet

feature engineering selesai dalam 206.3 detik
feat_train.parquet   1,243,617 baris  199 kolom  319 MB
feat_test.parquet      362,245 baris  198 kolom  93 MB


## 2. Pemeriksaan hasil feature engineering

Fitur hasil join bisa memiliki nilai kosong ketika sebuah pengiriman tidak punya baris di
tabel anak (mis. tanpa `route_segments`). LightGBM menangani NaN secara native, tetapi
proporsi NaN tetap diperiksa agar tidak ada fitur yang seluruhnya kosong.

In [3]:
sch = pl.read_parquet_schema(f_train)
print(f"{len(sch)} kolom fitur (termasuk {TARGET} dan shipment_id)")

null_info = (pl.scan_parquet(f_train)
             .select([pl.col(c).null_count().alias(c) for c in sch.names()])
             .collect().row(0, named=True))
miss = {c: v for c, v in null_info.items() if v > 0}
top_miss = sorted(miss.items(), key=lambda x: -x[1])[:10]
print(f"kolom dengan nilai kosong: {len(miss)} dari {len(sch)}")
n_tr = pl.scan_parquet(f_train).select(pl.len()).collect().item()
for c, v in top_miss:
    print(f"  {c:28s} {v:>9,} ({v/n_tr:.1%})")

199 kolom fitur (termasuk status_penanganan dan shipment_id)
kolom dengan nilai kosong: 18 dari 199
  prm_name                     1,032,408 (83.0%)
  prm_type                     1,032,408 (83.0%)
  prm_disc                     1,032,408 (83.0%)
  prm_start                    1,032,408 (83.0%)
  prm_end                      1,032,408 (83.0%)
  prm_month_offset             1,032,408 (83.0%)
  inv_disc_mean                  510,395 (41.0%)
  rs_dur_std                     463,180 (37.2%)
  gap_selesai                    412,323 (33.2%)
  q_upaya                        277,204 (22.3%)


## 3. Encoding kategorikal

- Kolom string diubah menjadi **kode integer** memakai `pd.factorize(sort=True)` pada
  gabungan train+test → hemat memori (int32 vs string object) dan bisa langsung dipakai
  sebagai *native categorical* LightGBM.
- `courier_id`, `hub_id`, `customer_id` **tidak** di-one-hot; ketiganya dipertahankan
  sebagai ID mentah untuk target encoding di notebook 04.
- Kolom numerik di-cast ke `float32` agar matriks 1,24 juta × 140 fitur muat di RAM.

In [4]:
preview_cols = ["shipment_id", "channel", "goods_category", "dest_region",
                "courier_id", "hub_id", TARGET]
tr_small = pd.read_parquet(f_train, columns=preview_cols).head(200_000)
te_small = pd.read_parquet(f_test, columns=[c for c in preview_cols if c != TARGET]).head(50_000)

# target tidak ikut di-encode: te_small memang tidak punya kolom target
tr_enc, te_enc, cat_cols, cat_categories = encode_categoricals(
    tr_small.drop(columns=[TARGET]), te_small)
print("kolom kategorikal:", cat_cols)
print("contoh kardinalitas:", {c: len(v) for c, v in cat_categories.items()})
display(tr_enc[["channel", "goods_category", "dest_region", "courier_id", "hub_id"]].head())

6 kolom kategorikal, 0 kolom numerik
kolom kategorikal: ['shipment_id', 'channel', 'goods_category', 'dest_region', 'courier_id', 'hub_id']
contoh kardinalitas: {'shipment_id': 250000, 'channel': 4, 'goods_category': 6, 'dest_region': 7, 'courier_id': 23806, 'hub_id': 654}


,channel,goods_category,dest_region,courier_id,hub_id
0,2,4,7,8885,406
1,1,5,6,17849,185
2,1,4,6,9402,164
3,1,4,2,22253,2
4,2,3,4,13802,635


## 4. Artefak yang dihasilkan

| Berkas | Isi |
|---|---|
| `data/processed/feat_train.parquet` | 1 baris per pengiriman train + 196 kolom fitur + target |
| `data/processed/feat_test.parquet` | kolom fitur yang sama untuk test |
| `data/processed/lgbm_final.txt` | model LightGBM final (dibuat di notebook 04) |
| `data/processed/preprocess.pkl` | kamus encoding + target encoding + pengali kelas |
| `data/processed/meta.json` | metrik validasi & versi library |

Daftar `DROP_COLS` (fitur dengan gain ≤ 0,01% pada uji importance) dibuang saat memuat
matriks di notebook model agar RAM cukup.

In [5]:
print(f"DROP_COLS: {len(DROP_COLS)} fitur dibuang saat pemodelan")
print("contoh:", DROP_COLS[:12])
print(f"kolom ID untuk target encoding: {TE_COLS}")

for p in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {p.name:24s} {p.stat().st_size/1e6:9.1f} MB")

DROP_COLS: 75 fitur dibuang saat pemodelan
contoh: ['prm_month_offset', 'pk_speed', 'moda_darat_n', 'ship_dow', 'hub_area', 'moda_laut_n', 'cus_kota_tipe', 'pickup_type', 'man_sum', 'co_contract', 'prm_disc', 'qty_items']
kolom ID untuk target encoding: ['courier_id', 'hub_id', 'customer_id']
  feat_test.parquet             92.8 MB
  feat_train.parquet           318.8 MB
